In [2]:
!pip install unsloth

In [3]:
!pip install unsloth vllm

In [4]:
import torch
from unsloth import FastLanguageModel

# SOLUTION 1: Increase max_seq_length to accommodate longer prompts
max_seq_length = 2048  # Increased from 1024 to 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6,  # You might need to reduce this if OOM
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-07 12:53:49 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-07 12:53:49 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.6.1: Fast Qwen2 patching. Transformers: 4.52.4. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-0.5b-instruct-bnb-4bit with actual GPU utilization = 59.43%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunke

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 06-07 12:54:33 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 06-07 12:54:34 [model_runner.py:1140] Model loading took 0.4672 GiB and 2.243003 seconds
INFO 06-07 12:54:38 [worker.py:287] Memory profiling takes 3.78 seconds
INFO 06-07 12:54:38 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.59) = 8.76GiB
INFO 06-07 12:54:38 [worker.py:287] model weights take 0.47GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.23GiB; the rest of the memory reserved for KV Cache is 7.04GiB.
INFO 06-07 12:54:39 [executor_base.py:112] # cuda blocks: 38444, # CPU blocks: 0
INFO 06-07 12:54:39 [executor_base.py:117] Maximum concurrency for 2048 tokens per request: 300.34x
INFO 06-07 12:54:39 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If ou

Capturing CUDA graph shapes:   0%|          | 0/31 [00:00<?, ?it/s]

INFO 06-07 12:55:26 [model_runner.py:1592] Graph capturing finished in 47 secs, took 0.43 GiB
INFO 06-07 12:55:26 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 52.07 seconds
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'k_norm', 'q_norm', 'post_feedforward_layernorm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'k_norm', 'q_norm', 'post_feedforward_layernorm']


Unsloth 2025.6.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [5]:
import re
from datasets import load_dataset, Dataset
import pandas as pd

# Load and prep dataset
SYSTEM_PROMPT = """
You will be given a HTML layout and a user request.
Your task is to add CSS (design) and JS (functionality) to the HTML layout.
First, think step by step about the thinking process in the designing and functionality that aligns with the user's request.
Then, provide the user with the answer inside <html> tag.

Respond strictly in the following format:
<think>
...
</think>

<html>
...
</html>
"""


# SOLUTION 2: Add prompt truncation function
def truncate_html_content(html_content, max_length=400):
    """Truncate HTML content if it's too long while preserving structure"""
    if len(html_content) <= max_length:
        return html_content

    # Try to truncate at a reasonable point
    truncated = html_content[:max_length]

    # Find the last complete tag or reasonable break point
    last_tag = truncated.rfind('>')
    last_space = truncated.rfind(' ')

    if last_tag > last_space and last_tag > max_length - 50:
        return truncated[:last_tag + 1] + "..."
    elif last_space > max_length - 50:
        return truncated[:last_space] + "..."
    else:
        return truncated + "..."

# SOLUTION 3: More concise prompt building
def build_prompt_concise(row):
    # Truncate HTML to prevent overly long prompts
    html_content = truncate_html_content(row['html'], 400)

    user_content = (
        f"Request: {row['query']}\n"
        f"HTML: {html_content}\n"
    )
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
        {'role': 'user', 'content': user_content}
    ]

# Load the data
df = pd.read_json('/content/drive/MyDrive/4학년 1학기/UI/ui_design_data.json')

# Construct prompts in chat-style format
def build_prompt(row):
    user_content = (
        f"Request: {row['query']}\n"
        f"HTML: {row['html']}\n"
    )
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
        {'role': 'user', 'content': user_content}
    ]

df['prompt'] = df.apply(build_prompt, axis=1)

# SOLUTION 4: Filter out prompts that are still too long
def check_prompt_length(prompt_list, max_length=1500):
    """Check if tokenized prompt length is within limits"""
    text = tokenizer.apply_chat_template(prompt_list, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    return len(tokens) <= max_length

# Filter dataset to only include prompts within length limits
df['prompt_valid'] = df['prompt'].apply(lambda x: check_prompt_length(x, 1500))
df_filtered = df[df['prompt_valid']].copy()

print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(df_filtered)}")

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_filtered[['prompt']])

# SOLUTION 5: Adjust training configuration for longer sequences
max_prompt_length = 2048  # Increased from 1024
max_completion_length = 2048 #max_seq_length - max_prompt_length


Original dataset size: 6800
Filtered dataset size: 6799


In [6]:
!pip install colorspacious

In [7]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,  # Increased for longer sequences
    num_generations = 4,  # Reduced to save memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 250,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "wandb",
    output_dir = "/content/drive/MyDrive/4학년 1학기/UI",
    log_completions = True,
    # wandb_log_unique_prompts = True,
)

# Reward functions (keeping your original ones)
def format_reward_func(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think><html>.*?</html>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content, re.DOTALL) for content in completion_contents]
    return [1.0 if match else 0.0 for match in matches]

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


In [8]:
import re
import colorsys
import math
from typing import List, Tuple, Dict, Optional
from colorspacious import cspace_convert, deltaE

def hex_to_rgb(hex_color: str) -> Tuple[int, int, int]:
    """Convert hex color to RGB tuple"""
    hex_color = hex_color.lstrip('#')
    if len(hex_color) == 3:
        hex_color = ''.join([c*2 for c in hex_color])
    try:
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    except (ValueError, IndexError):
        return (128, 128, 128)  # Default gray if parsing fails

def rgb_to_hsl(r: int, g: int, b: int) -> Tuple[float, float, float]:
    """Convert RGB to HSL"""
    r, g, b = r/255.0, g/255.0, b/255.0
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return (h * 360, s * 100, l * 100)  # H in degrees, S/L in percentages

def calculate_contrast_ratio(color1: Tuple[int, int, int], color2: Tuple[int, int, int]) -> float:
    """Calculate WCAG contrast ratio between two colors"""
    def luminance(r, g, b):
        def adjust(c):
            c = c / 255.0
            return c / 12.92 if c <= 0.03928 else pow((c + 0.055) / 1.055, 2.4)
        return 0.2126 * adjust(r) + 0.7152 * adjust(g) + 0.0722 * adjust(b)

    lum1 = luminance(*color1)
    lum2 = luminance(*color2)

    brighter = max(lum1, lum2)
    darker = min(lum1, lum2)

    return (brighter + 0.05) / (darker + 0.05)

def calculate_delta_e(color1: Tuple[int, int, int], color2: Tuple[int, int, int]) -> float:
    """Calculate Delta E (CIE2000) color difference"""
    try:
        lab1 = cspace_convert(color1, "sRGB1", "CIELab")
        lab2 = cspace_convert(color2, "sRGB1", "CIELab")
        return deltaE(lab1, lab2, input_space="CIELab")
    except:
        # Fallback to simple Euclidean distance in RGB space
        return math.sqrt(sum((a - b) ** 2 for a, b in zip(color1, color2))) / 441.67

def extract_colors_from_css(css_content: str) -> Dict[str, List[Tuple[int, int, int]]]:
    """Extract colors from CSS content and categorize them"""
    # Color patterns
    hex_pattern = r'#([0-9a-fA-F]{3}|[0-9a-fA-F]{6})\b'
    rgb_pattern = r'rgb\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)'
    rgba_pattern = r'rgba\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*[\d.]+\s*\)'

    colors = {
        'background': [],
        'text': [],
        'border': [],
        'all': []
    }

    # Extract hex colors
    hex_matches = re.findall(hex_pattern, css_content)
    for hex_color in hex_matches:
        rgb = hex_to_rgb('#' + hex_color)
        colors['all'].append(rgb)

    # Extract RGB colors
    rgb_matches = re.findall(rgb_pattern, css_content)
    for r, g, b in rgb_matches:
        rgb = (int(r), int(g), int(b))
        colors['all'].append(rgb)

    # Extract RGBA colors
    rgba_matches = re.findall(rgba_pattern, css_content)
    for r, g, b in rgba_matches:
        rgb = (int(r), int(g), int(b))
        colors['all'].append(rgb)

    # Categorize colors based on CSS properties
    for prop in ['background-color', 'background']:
        pattern = rf'{prop}\s*:\s*([^;]+)'
        matches = re.findall(pattern, css_content, re.IGNORECASE)
        for match in matches:
            hex_in_prop = re.findall(hex_pattern, match)
            for hex_color in hex_in_prop:
                colors['background'].append(hex_to_rgb('#' + hex_color))

    for prop in ['color']:
        pattern = rf'{prop}\s*:\s*([^;]+)'
        matches = re.findall(pattern, css_content, re.IGNORECASE)
        for match in matches:
            hex_in_prop = re.findall(hex_pattern, match)
            for hex_color in hex_in_prop:
                colors['text'].append(hex_to_rgb('#' + hex_color))

    for prop in ['border-color', 'border']:
        pattern = rf'{prop}\s*:\s*([^;]+)'
        matches = re.findall(pattern, css_content, re.IGNORECASE)
        for match in matches:
            hex_in_prop = re.findall(hex_pattern, match)
            for hex_color in hex_in_prop:
                colors['border'].append(hex_to_rgb('#' + hex_color))

    return colors

def extract_css_from_response(response: str) -> str:
    """Extract CSS content from response"""
    # Look for CSS in <style> tags
    style_match = re.search(r'<style[^>]*>(.*?)</style>', response, re.DOTALL | re.IGNORECASE)
    if style_match:
        return style_match.group(1)

    # Look for CSS in answer tags
    answer_match = re.search(r'<answer>(.*?)</answer>', response, re.DOTALL | re.IGNORECASE)
    if answer_match:
        answer_content = answer_match.group(1)
        style_in_answer = re.search(r'<style[^>]*>(.*?)</style>', answer_content, re.DOTALL | re.IGNORECASE)
        if style_in_answer:
            return style_in_answer.group(1)

    return response  # Return full response if no specific CSS found

# Reward Functions

def contrast_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for text contrast compliance"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0
        total_checks = 0

        # Check text-background contrast
        if colors['text'] and colors['background']:
            for text_color in colors['text']:
                for bg_color in colors['background']:
                    contrast_ratio = calculate_contrast_ratio(text_color, bg_color)

                    # Normal text contrast (≥ 4.5:1)
                    if contrast_ratio >= 4.5:
                        score += 1.0
                    elif contrast_ratio >= 3.0:  # Large text contrast (≥ 3:1)
                        score += 0.7
                    elif contrast_ratio >= 2.0:
                        score += 0.3

                    total_checks += 1

        # If no explicit text/background colors found, check general color pairs
        if total_checks == 0 and len(colors['all']) >= 2:
            for i, color1 in enumerate(colors['all']):
                for color2 in colors['all'][i+1:]:
                    contrast_ratio = calculate_contrast_ratio(color1, color2)
                    if contrast_ratio >= 4.5:
                        score += 1.0
                    elif contrast_ratio >= 3.0:
                        score += 0.7
                    total_checks += 1
                    if total_checks >= 3:  # Limit checks to avoid too many comparisons
                        break
                if total_checks >= 3:
                    break

        final_score = score / max(total_checks, 1) if total_checks > 0 else 0.5
        scores.append(min(1.0, final_score))

    return scores

def color_distance_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for color distance (Delta E)"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0
        total_pairs = 0

        if len(colors['all']) >= 2:
            for i, color1 in enumerate(colors['all']):
                for color2 in colors['all'][i+1:]:
                    delta_e = calculate_delta_e(color1, color2)

                    # Reward higher for Delta E ≥ 10
                    if delta_e >= 10:
                        score += 1.0
                    elif delta_e >= 5:
                        score += 0.6
                    elif delta_e >= 2:
                        score += 0.3

                    total_pairs += 1
                    if total_pairs >= 5:  # Limit to avoid too many comparisons
                        break
                if total_pairs >= 5:
                    break

        final_score = score / max(total_pairs, 1) if total_pairs > 0 else 0.5
        scores.append(min(1.0, final_score))

    return scores

def saturation_range_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for saturation range compliance"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0
        total_colors = 0

        for rgb in colors['all']:
            h, s, l = rgb_to_hsl(*rgb)

            # Emphasis colors should have saturation within [30%, 90%]
            if 30 <= s <= 90:
                score += 1.0
            elif 20 <= s <= 95:  # Close to ideal range
                score += 0.7
            elif s > 10:  # At least some saturation
                score += 0.3

            total_colors += 1

        final_score = score / max(total_colors, 1) if total_colors > 0 else 0.5
        scores.append(min(1.0, final_score))

    return scores

def lightness_range_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for lightness range compliance"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0
        total_colors = 0

        for rgb in colors['all']:
            h, s, l = rgb_to_hsl(*rgb)

            # Ideal lightness range: avoid too dark (<20) or too light (>90)
            if 20 <= l <= 90:
                score += 1.0
            elif 10 <= l <= 95:  # Acceptable range
                score += 0.7
            elif 5 <= l <= 98:   # Barely acceptable
                score += 0.4
            else:
                score += 0.1  # Poor lightness

            total_colors += 1

        final_score = score / max(total_colors, 1) if total_colors > 0 else 0.5
        scores.append(min(1.0, final_score))

    return scores


def color_harmony_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for color harmony assessment"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0

        if len(colors['all']) >= 2:
            hues = []
            for rgb in colors['all']:
                h, s, l = rgb_to_hsl(*rgb)
                hues.append(h)

            # Check for color harmony patterns
            hues.sort()

            # Monochromatic (similar hues)
            hue_range = max(hues) - min(hues)
            if hue_range <= 30:  # Within 30 degrees
                score += 0.8

            # Analogous (adjacent hues, 30-90 degrees apart)
            elif 30 <= hue_range <= 90:
                score += 1.0

            # Complementary (opposite hues, ~180 degrees apart)
            elif len(hues) >= 2:
                for i, hue1 in enumerate(hues):
                    for hue2 in hues[i+1:]:
                        hue_diff = abs(hue1 - hue2)
                        hue_diff = min(hue_diff, 360 - hue_diff)  # Handle circular nature

                        if 150 <= hue_diff <= 210:  # Near complementary
                            score += 1.0
                            break
                        elif 120 <= hue_diff <= 240:  # Triadic or split-complementary
                            score += 0.8
                            break

            # Triadic (120 degrees apart)
            if len(hues) >= 3:
                triadic_score = 0
                for i in range(len(hues) - 2):
                    diff1 = abs(hues[i+1] - hues[i])
                    diff2 = abs(hues[i+2] - hues[i+1])

                    if 100 <= diff1 <= 140 and 100 <= diff2 <= 140:
                        triadic_score = 0.9
                        break

                score = max(score, triadic_score)

        else:
            score = 0.3  # Minimal score for single color

        scores.append(min(1.0, score))

    return scores

def aesthetic_balance_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for overall aesthetic balance"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)
        colors = extract_colors_from_css(css_content)

        score = 0.0
        factors = 0

        if colors['all']:
            # Color count balance (not too many, not too few)
            color_count = len(set(colors['all']))  # Unique colors
            if 2 <= color_count <= 5:
                score += 1.0
                factors += 1
            elif color_count == 1 or 6 <= color_count <= 8:
                score += 0.6
                factors += 1
            else:
                score += 0.2
                factors += 1

            # Saturation balance
            saturations = [rgb_to_hsl(*rgb)[1] for rgb in colors['all']]
            saturation_range = max(saturations) - min(saturations)

            if 20 <= saturation_range <= 60:  # Good variety without extremes
                score += 1.0
                factors += 1
            elif saturation_range <= 80:
                score += 0.7
                factors += 1
            else:
                score += 0.3
                factors += 1

            # Lightness balance
            lightnesses = [rgb_to_hsl(*rgb)[2] for rgb in colors['all']]
            lightness_range = max(lightnesses) - min(lightnesses)

            if 30 <= lightness_range <= 70:  # Good contrast without extremes
                score += 1.0
                factors += 1
            elif lightness_range >= 20:
                score += 0.7
                factors += 1
            else:
                score += 0.3
                factors += 1

        final_score = score / max(factors, 1) if factors > 0 else 0.5
        scores.append(min(1.0, final_score))

    return scores

def css_completeness_reward_func(completions, **kwargs) -> List[float]:
    """Reward function for CSS completeness and structure"""
    scores = []

    for completion in completions:
        response = completion[0]['content']
        css_content = extract_css_from_response(response)

        score = 0.0

        # Check for basic CSS structure
        if '{' in css_content and '}' in css_content:
            score += 0.3

        # Check for color properties
        color_properties = ['color', 'background-color', 'background', 'border-color']
        for prop in color_properties:
            if prop in css_content.lower():
                score += 0.2

        # Check for responsive design elements
        responsive_elements = ['@media', 'max-width', 'min-width', '%', 'rem', 'em']
        for element in responsive_elements:
            if element in css_content.lower():
                score += 0.1

        # Check for interactive elements
        interactive_elements = [':hover', ':focus', ':active', 'transition', 'transform']
        for element in interactive_elements:
            if element in css_content.lower():
                score += 0.1

        scores.append(min(1.0, score))

    return scores

# Combined reward function that weights all criteria
def combined_design_reward_func(completions, **kwargs) -> List[float]:
    """Combined reward function with weighted criteria"""

    # Get individual scores
    contrast_scores = contrast_reward_func(completions, **kwargs)
    distance_scores = color_distance_reward_func(completions, **kwargs)
    saturation_scores = saturation_range_reward_func(completions, **kwargs)
    lightness_scores = lightness_range_reward_func(completions, **kwargs)
    harmony_scores = color_harmony_reward_func(completions, **kwargs)
    aesthetic_scores = aesthetic_balance_reward_func(completions, **kwargs)
    completeness_scores = css_completeness_reward_func(completions, **kwargs)

    # Weight the different aspects
    weights = {
        'contrast': 0.25,      # Accessibility is crucial
        'distance': 0.15,      # Color differentiation
        'saturation': 0.15,    # Color vibrancy
        'lightness': 0.15,     # Visual balance
        'harmony': 0.20,       # Aesthetic coherence
        'aesthetic': 0.15,     # Overall balance
        'completeness': 0.10   # Technical completeness
    }

    combined_scores = []
    for i in range(len(completions)):
        combined_score = (
            contrast_scores[i] * weights['contrast'] +
            distance_scores[i] * weights['distance'] +
            saturation_scores[i] * weights['saturation'] +
            lightness_scores[i] * weights['lightness'] +
            harmony_scores[i] * weights['harmony'] +
            aesthetic_scores[i] * weights['aesthetic'] +
            completeness_scores[i] * weights['completeness']
        )
        combined_scores.append(combined_score)

    return combined_scores

def count_xml(text) -> float:
    count = 0.0
    if text.count("<think>\n") == 1:
        count += 0.125
    if text.count("\n</think>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

In [9]:
!pip install pyppeteer
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()


In [10]:
import asyncio
from pyppeteer import launch
from typing import List

AXE_SCRIPT_URL = "https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.7.2/axe.min.js"

async def _run_axe_on_html(html: str) -> dict:
    browser = await launch(headless=True, args=[
    '--no-sandbox',
    '--disable-setuid-sandbox',
    '--disable-gpu',
    '--disable-dev-shm-usage'
])
    page = await browser.newPage()
    await page.setContent(html)
    await page.addScriptTag({'url': AXE_SCRIPT_URL})

    result = await page.evaluate('''async () => {
        return await axe.run(document, {
            runOnly: {
                type: 'tag',
                values: ['wcag2aa']
            },
            resultTypes: ['violations'],
            reporter: 'v2'
        });
    }''')

    await browser.close()
    return result

def axe_violation_reward_func(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    """
    Run axe-core on each HTML completion and score accessibility violations.
    Input:
      - completions: list of HTML strings
    Output:
      - list of floats, each between 0 and 1, higher is better
    """
    impact_weights = {
        'minor': 0.05,
        'moderate': 0.1,
        'serious': 0.4,
        'critical': 0.5
    }

    async def run_all(completions):
        results = []
        for i, html in enumerate(completions):
            try:
                res = await _run_axe_on_html(html)
                results.append(res)
            except Exception as e:
                msg = f"[axe] Error {i}: {e}"
                print(msg)
                results.append({'violations': []})
                with open("/content/axe.log", "a") as f:
                    f.write(msg + "\n")
                    f.write("no violations" + "\n")

        return results

    # Run pyppeteer event loop synchronously
    axe_results = asyncio.get_event_loop().run_until_complete(run_all(completions))

    scores = []
    for i, result in enumerate(axe_results):
        try:
            violations = result.get('violations', [])
            penalty = 0.0
            for v in violations:
                impact = v.get('impact', 'minor')
                weight = impact_weights.get(impact, 0.05)
                penalty += weight * len(v.get('nodes', []))
            penalty = min(penalty, 1.0)
            score = max(1.0 - penalty, 0.0)
            scores.append(score)
        except Exception as e:
            print(f"[axe_violation_reward_func] Error calculating score for completion {i}: {e}")
            scores.append(0.0)

    return scores

In [2]:
!ls


drive			 huggingface_tokenizers_cache  unsloth_compiled_cache
grpo_trainer_lora_model  sample_data		       wandb
drive			 huggingface_tokenizers_cache  unsloth_compiled_cache
grpo_trainer_lora_model  sample_data		       wandb


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        # format_reward_func,
        # combined_design_reward_func,
        axe_violation_reward_func,
    ],
    args = training_args,
    train_dataset = train_dataset,
    wandb_log_unique_prompts = True,
)

trainer.train()

# INFERENCE - Updated to use the correct format
model.save_lora("grpo_saved_lora")

# Test inference
text = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Calculate pi."},
], tokenize=False, add_generation_prompt=True)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 512,  # Reduced to ensure total length fits
)

output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

print("Generated output:")
print(output)